# Notatki Hipotezy 

**Sieci uwagowe:** Fz, F3, F4, F7, F8, Cz, Pz, P3, P4, T5, T6

**DMN:** Fp1, Fp2, Fz, Pz, P3, P4, T5, T6 (oś Fz↔Pz najważniejsza)

**Emocje:** Fp1, Fp2, F3, F4, F7, F8, Fz, T3, T4, Cz, Pz, O1, O2

## Czego się spodziewać w paśmie 4–15 Hz
Theta (4–7) i alfę (8–12) — to  najbardziej diagnostyczne pasma dla emocji i uwagi. Łapiesz też dolną krawędź beta/SMR (13–15), ale wyższa beta i gamma odpadają, co przy biernym oglądaniu nie jest dużą stratą.

### Theta (4–7 Hz):

FMθ na Fz — kluczowy marker zaangażowania emocjonalnego i regulacji. Wzrost dla treści nasyconych emocjonalnie (zarówno pozytywnych jak negatywnych) vs neutralnych.
Theta na osi czołowo-tylnej (Fz↔Pz, F3/F4↔P3/P4) — w DTF spodziewałbym się wzrostu przepływu z czoła ku tyłowi przy materiale emocjonalnym (kontrola top-down nad uwagą wzrokową).
Theta na bocznych czołowo-skroniowych (F7/F8, T3/T4) — często silniejsza dla emocji negatywnych.

### Alfa (8–12 Hz)

Asymetria czołowa alfa (F3/F4, F7/F8) — najtwardszy marker walencji w EEG. Indeks log(α_prawe) − log(α_lewe) na parze F4/F3 lub F8/F7. Pozytywne → względnie mniej alfy lewostronnie, negatywne → odwrotnie. Efekt wyraźniejszy w dolnej alfie (8–10 Hz).
Alfa potyliczno-ciemieniowa (O1, O2, P3, P4, Pz) — najczystszy wskaźnik pobudzenia. Desynchronizacja (spadek mocy) przy wyższym arousal, niezależnie od znaku emocji. To prawdopodobnie Twój najlepszy sygnał arousal.
Rytm mu (C3, Cz, C4, ~10 Hz) — istotny przy bajkach: supresja mu odzwierciedla zaangażowanie systemu obserwacji działań. Bajki z wyrazistymi postaciami/akcją powinny dawać silniejszą supresję.
Warto rozdzielić alfę dolną (8–10 Hz, arousal) i górną (10–12 Hz, uwaga semantyczna) — często reagują w przeciwnych kierunkach.

### Niska beta / SMR (13–15 Hz)
Marginalna w Twoim paśmie, ale na centralno-czołowych elektrodach (Cz, C3, C4, Fz) wzrost przy niepokoju/lęku. Efekt słabszy, lecz wykrywalny przy odpowiedniej liczbie prób.
Przewidywania dla manipulacji walencja × arousal
KontrastNajbardziej diagnostyczne kanały i pasmoPozytywne vs negatywne (walencja)F3/F4, F7/F8 — asymetria alfy (8–10 Hz)Wysokie vs niskie pobudzenie (arousal)O1, O2, P3, P4, Pz — desynchronizacja alfy; Fz — thetaEmocjonalne vs neutralneFz — FMθ; F7/F8, T3/T4 — theta lateralnaLęk/stres (jeśli wystąpi)Fz, Cz — niska beta; F4, F8 — prawostronna asymetria alfy

---
**Sugestia dla DTF**

Walencja często różnicuje lateralizację efektów czołowych, a kontrast emocjonalne-vs-neutralne jest bardziej aksjalny (oś Fz–Pz). Jeśli liczysz kierunkowe przepływy, sprawdziłbym osobno: (a) Fz↔Pz/P3/P4 w theta — top-down regulacja, oraz (b) F3↔F4 i F7↔F8 w alfie — międzypółkulowe wzorce walencyjne. Dla 1-minutowych fragmentów masz przy 128 Hz ~7680 próbek na bodziec, co dla MVAR rzędu ~6 (BIC) jest już komfortowe.

In [ ]:
import os
import sys
import importlib
import json
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

sys.path.insert(0, os.path.abspath('..'))

# Reload local src modules so that changes on disk are always picked up
# without requiring a full kernel restart.
import src.mtmvar
import src.export
importlib.reload(src.mtmvar)
importlib.reload(src.export)

from src.mtmvar import (
    full_freq_dtf,
    multivariate_spectra,
    mvar_plot,
    mvar_criterion,
    compute_and_plot_mvar,
)
from src.export import load_eeg_signals, plot_loaded_eeg_signals
from src.export import get_export_metadata, load_xarray_from_netcdf


In [ ]:
# ── Folders ──────────────────────────────────────────────────────────────────
export_folder = Path("/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported")
figures_folder = Path("../../EEG_DTF_figures")
figures_folder.mkdir(exist_ok=True)
# ── Output directory (relative to this notebook's location) ──────────────────
FIGURES_DIR  = figures_folder
# ── Run mode ─────────────────────────────────────────────────────────────────
# smoke_test=True -> process only first N dyads found in the export folder.
# smoke_test=False -> process all dyads.
smoke_test = False#True
smoke_dyads_n = 10

# ── Events to analyse ────────────────────────────────────────────────────────
TARGET_EVENTS = ['Brave', 'Peppa', 'Incredibles']

# ── Channel subset ───────────────────────────────────────────────────────────
# 21-channel MVAR is feasible but slow. Set to None to use all channels.
CHANNEL_SUBSET = None # ['F3', 'Fz', 'F4', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4']

# ── MVAR settings ────────────────────────────────────────────────────────────
PLOT_CRITERION_CURVES = False  # whether to plot AIC/HQ/SC curves for model order selection
MAX_MODEL_ORDER = 15          # upper bound for AIC model-order search
OPTIMAL_MODEL_ORDER = 13    # set to an int to skip the AIC search
CRIT_TYPE = 'HQ' #'AIC', 'HQ', or 'SC'.

# ── Optional pre-filtering settings (applied before margin trimming) ─────────
# Set to None to disable a specific filter stage.
# SIGNAL_LOW_CUTOFF_HZ  -> high-pass cutoff frequency.
# SIGNAL_HIGH_CUTOFF_HZ -> low-pass cutoff frequency.
SIGNAL_LOW_CUTOFF_HZ = None #1.0
SIGNAL_HIGH_CUTOFF_HZ = None #45.0

# ── Frequency axis ───────────────────────────────────────────────────────────
FREQ_MIN = 4    # Hz
FREQ_MAX = 20   # Hz
FREQ_STEP = 0.5   # Hz


In [ ]:
# Crawl export folder for EEG ncdf files matching the target events
all_eeg_files = sorted([
    p for p in export_folder.rglob("*.nc")
    if "_EEG_" in p.name
    and any(p.stem.endswith(f"_{ev}") for ev in TARGET_EVENTS)
])

if not all_eeg_files:
    raise FileNotFoundError(f"No EEG NetCDF files found for events {TARGET_EVENTS} under: {export_folder}")

# Group files by dyad (e.g. W_030)
files_by_dyad = {}
for p in all_eeg_files:
    parts = p.stem.split('_')
    dyad_id = f"{parts[0]}_{parts[1]}" if len(parts) >= 2 else p.stem
    files_by_dyad.setdefault(dyad_id, []).append(p)

all_dyads = sorted(files_by_dyad.keys())

if smoke_test:
    dyads_to_process = all_dyads[:smoke_dyads_n]
    mode = f"SMOKE TEST (first {smoke_dyads_n} dyads)"
else:
    dyads_to_process = all_dyads
    mode = "FULL ANALYSIS"

eeg_files = []
for dyad in dyads_to_process:
    eeg_files.extend(sorted(files_by_dyad[dyad]))

print(f"Mode: {mode}")
print(f"Dyads selected: {len(dyads_to_process)} / {len(all_dyads)}")
print(f"Files selected: {len(eeg_files)} / {len(all_eeg_files)}")
print("Dyads:")
for dyad in dyads_to_process:
    print(f"  - {dyad}")


In [ ]:
# load_eeg_signals, plot_loaded_eeg_signals  -> src/export.py
# compute_and_plot_mvar                      -> src/mtmvar.py
# All three are imported in cell 1.


In [ ]:
results = {}
failed = []

for ncdf_path in eeg_files:
    label = ncdf_path.stem   # e.g. W_030_EEG_cg_Brave
    display(Markdown(f"### {label}"))
    print(f"Processing: {ncdf_path.name}")

    try:
        da = load_xarray_from_netcdf(str(ncdf_path))
        meta = get_export_metadata(da)
        ff_dtf, spectra, chan_names, crit, model_order_range, p_opt = compute_and_plot_mvar(
            ncdf_path,
            channel_subset=CHANNEL_SUBSET,
            max_model_order=MAX_MODEL_ORDER,
            optimal_model_order=OPTIMAL_MODEL_ORDER,
            crit_type=CRIT_TYPE,
            freq_min=FREQ_MIN,
            freq_max=FREQ_MAX,
            freq_step=FREQ_STEP,
            low_cutoff_hz=SIGNAL_LOW_CUTOFF_HZ,
            high_cutoff_hz=SIGNAL_HIGH_CUTOFF_HZ,
            plot=False,
            plot_loaded_signal=False
        )
        results[label] = {'ff_dtf': ff_dtf, 'spectra': spectra, 'chan_names': chan_names, 'crit': crit, 'model_order_range': model_order_range, 'p_opt': p_opt, 'metadata': meta}
        print(f"  ✓ Done: {label}")
    except Exception as exc:
        failed.append((label, str(exc)))
        print(f"  ✗ FAILED: {exc}")

print(f"\n{'='*60}")
print(f"Finished. OK: {len(results)}, Failed: {len(failed)}")
if failed:
    print("Failed files:")
    for name, err in failed:
        print(f"  - {name}: {err}")


In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

# ── Layout constants ──────────────────────────────────────────────────────────
MOVIE_ORDER  = ['Peppa', 'Brave', 'Incredibles']
ROLE_ORDER   = ['ch', 'cg']
ROLE_LABELS  = {'ch': 'Child', 'cg': 'Caregiver'}




# ── Helper: draw one MVAR panel into a SubplotSpec ───────────────────────────
def _draw_mvar_panel(fig, subplot_spec, ff_dtf, spectra, freqs, chan_names, scale='linear',
                     global_max_spectra=None, global_max_dtf=None):
    """Render MVAR spectra (diagonal) + ffDTF (off-diagonal) into a SubplotSpec."""
    on_diag  = np.abs(spectra.copy())
    off_diag = np.abs(ff_dtf.copy())

    if scale == 'sqrt':
        on_diag  = np.sqrt(on_diag)
        off_diag = np.sqrt(off_diag)
    elif scale == 'log':
        on_diag  = np.log(on_diag  + 1e-12)
        off_diag = np.log(off_diag + 1e-12)

    n_chan = on_diag.shape[0]

    # Zero-out irrelevant parts (same convention as mvar_plot)
    for i in range(n_chan):
        for j in range(n_chan):
            if i != j:
                on_diag[i, j, :] = 0
            else:
                off_diag[i, i, :] = 0

    max_on_diag  = global_max_spectra if global_max_spectra is not None else np.max(on_diag)
    max_off_diag = global_max_dtf     if global_max_dtf     is not None else np.max(off_diag)

    inner_gs = GridSpecFromSubplotSpec(
        n_chan, n_chan, subplot_spec=subplot_spec, wspace=0, hspace=0
    )

    for i in range(n_chan):
        for j in range(n_chan):
            ax = fig.add_subplot(inner_gs[i, j])

            if i != j:
                y = np.real(off_diag[i, j, :])
                ax.plot(freqs, y, lw=0.5)
                ax.fill_between(freqs, y, 0, color='skyblue', alpha=0.4)
                ax.set_ylim([0, max_off_diag])
            else:
                y = np.real(on_diag[i, j, :])
                ax.plot(freqs, y, color=[0.7, 0.7, 0.7], lw=0.5)
                ax.fill_between(freqs, y, 0, color=[0.7, 0.7, 0.7], alpha=0.4)
                ax.set_ylim([0, max_on_diag])

            ax.tick_params(
                labelleft=(j == 0), labelbottom=(i == n_chan - 1),
                left=(j == 0), bottom=(i == n_chan - 1),
                labelsize=4,
            )
            if i == n_chan - 1:
                ax.set_xlabel(chan_names[j], fontsize=4)
            if j == 0:
                ax.set_ylabel(chan_names[i], fontsize=4)


# ── Main: composite figure per dyad ──────────────────────────────────────────
def make_composite_per_dyad(dyad_id, results, freq_min, freq_max, freq_step,
                             scale='linear', output_dir=None, dpi=300,
                             global_max_spectra=None, global_max_dtf=None):
    """
    Create and save a 2×3 composite MVAR figure for one dyad.

    Layout
    ------
    Rows    : child (top), caregiver (bottom)
    Columns : Peppa, Brave, Incredibles  (left → right)

    global_max_spectra / global_max_dtf
        When provided (pre-computed across all dyads), all panels share the
        same colour/amplitude scale, making cross-dyad comparison possible.
    """
    freqs = np.arange(freq_min, freq_max + freq_step, freq_step)

    # Build (role, movie) → result lookup
    lookup = {}
    for label, data in results.items():
        parts = label.split('_')
        # Stem format: W_030_EEG_{role}_{movie}
        if len(parts) >= 5 and f"{parts[0]}_{parts[1]}" == dyad_id:
            role  = parts[3]   # 'ch' or 'cg'
            movie = parts[4]   # 'Peppa', 'Brave', 'Incredibles'
            lookup[(role, movie)] = data

    # Figure: outer GridSpec with a thin header row and a thin label column
    fig = plt.figure(figsize=(20, 14), dpi=dpi)
    outer_gs = GridSpec(
        3, 4,
        figure=fig,
        height_ratios=[0.05, 1, 1],
        width_ratios=[0.04, 1, 1, 1],
        left=0.05, right=0.99,
        top=0.93, bottom=0.04,
        wspace=0.04, hspace=0.06,
    )

    # Column headers (row 0, cols 1-3)
    for col, movie in enumerate(MOVIE_ORDER):
        ax_hdr = fig.add_subplot(outer_gs[0, col + 1])
        ax_hdr.text(0.5, 0.5, movie, ha='center', va='center',
                    fontsize=14, fontweight='bold', transform=ax_hdr.transAxes)
        ax_hdr.axis('off')

    # Row labels (rows 1-2, col 0)
    for row, role in enumerate(ROLE_ORDER):
        ax_lbl = fig.add_subplot(outer_gs[row + 1, 0])
        ax_lbl.text(0.5, 0.5, ROLE_LABELS[role], ha='center', va='center',
                    fontsize=12, fontweight='bold', rotation=90,
                    transform=ax_lbl.transAxes)
        ax_lbl.axis('off')

    # MVAR panels (rows 1-2, cols 1-3)
    for row, role in enumerate(ROLE_ORDER):
        for col, movie in enumerate(MOVIE_ORDER):
            panel_spec = outer_gs[row + 1, col + 1]
            key = (role, movie)
            if key in lookup:
                data = lookup[key]
                _draw_mvar_panel(
                    fig, panel_spec,
                    data['ff_dtf'], data['spectra'], freqs,
                    data['chan_names'], scale=scale,
                    global_max_spectra=global_max_spectra,
                    global_max_dtf=global_max_dtf,
                )
            else:
                ax_na = fig.add_subplot(panel_spec)
                ax_na.text(0.5, 0.5, 'N/A', ha='center', va='center',
                           fontsize=14, color='gray', transform=ax_na.transAxes)
                ax_na.set_facecolor('#f0f0f0')
                ax_na.axis('off')

    fig.suptitle(f"MVAR Connectivity — Dyad {dyad_id}, group {data['metadata']['child_info']['group']}", fontsize=16, fontweight='bold', y=0.98)

    if output_dir is not None:
        out_path = Path(output_dir) / f"{dyad_id}_MVAR_composite.png"
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=dpi, bbox_inches='tight')
        print(f"Saved → {out_path}")

    plt.show()
    plt.close(fig)
    return 


# ── Run ───────────────────────────────────────────────────────────────────────
result_dyads = sorted({
    f"{lbl.split('_')[0]}_{lbl.split('_')[1]}"
    for lbl in results
})

# Plot criterion curves for all successfully processed files
if PLOT_CRITERION_CURVES:
    plt.figure(figsize=(6, 4))
    for label, data in results.items():
        crit = data['crit']
        model_order_range = data['model_order_range']
        p_opt = data['p_opt']

        plt.plot(model_order_range, crit, marker='o')
        plt.axvline(p_opt, color='red', linestyle='--', label=f"Optimal p={p_opt}")
        plt.title(f"Model Order Selection ({label})")
        plt.xlabel("Model Order (p)")
        plt.ylabel(CRIT_TYPE)
        plt.xticks(model_order_range)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# ── Compute global amplitude scales across ALL dyads/roles/movies ─────────────
# Maxima are restricted to the plotted frequency range [FREQ_MIN, FREQ_MAX].
_scale = 'linear'  # must match the scale= argument passed below
_plot_freqs = np.arange(FREQ_MIN, FREQ_MAX + FREQ_STEP, FREQ_STEP)

def _apply_scale(arr, scale):
    arr = np.abs(arr)
    if scale == 'sqrt':
        return np.sqrt(arr)
    elif scale == 'log':
        return np.log(arr + 1e-12)
    return arr

global_max_spectra = 0.0
global_max_dtf     = 0.0

for data in results.values():
    on_diag  = _apply_scale(data['spectra'], _scale)
    off_diag = _apply_scale(data['ff_dtf'],  _scale)

    # Build a frequency mask aligned to this result's frequency axis.
    # The stored arrays were computed on np.arange(FREQ_MIN, FREQ_MAX+FREQ_STEP, FREQ_STEP),
    # so the mask will be all-True in the normal case; it protects against
    # arrays that span a wider range.
    n_freq = on_diag.shape[-1]
    stored_freqs = np.linspace(_plot_freqs[0], _plot_freqs[-1], n_freq) \
        if n_freq == len(_plot_freqs) \
        else np.arange(0, n_freq) * (FREQ_STEP)   # fallback: use index positions
    # Prefer explicit reconstruction from stored shape vs _plot_freqs match
    freq_mask = (_plot_freqs >= FREQ_MIN) & (_plot_freqs <= FREQ_MAX)  # always True for _plot_freqs itself

    n = on_diag.shape[0]
    # spectra: diagonal only, within frequency mask
    for i in range(n):
        global_max_spectra = max(global_max_spectra, float(np.max(on_diag[i, i, freq_mask])))
    # DTF: off-diagonal only, within frequency mask
    for i in range(n):
        for j in range(n):
            if i != j:
                global_max_dtf = max(global_max_dtf, float(np.max(off_diag[i, j, freq_mask])))
global_max_spectra = 10.0
global_max_dtf     = 0.02
print(f"Global scale — spectra max: {global_max_spectra:.4g},  DTF max: {global_max_dtf:.4g}")
print(f"  (computed over {freq_mask.sum()} frequency bins: {FREQ_MIN}–{FREQ_MAX} Hz)")

# Plot composite figures for all dyads with at least one successful result
print(f"Building composite figures for {len(result_dyads)} dyad(s): {result_dyads}")

for dyad in result_dyads:
    make_composite_per_dyad(
        dyad_id=dyad,
        results=results,
        freq_min=FREQ_MIN,
        freq_max=FREQ_MAX,
        freq_step=FREQ_STEP,
        scale=_scale,
        output_dir=FIGURES_DIR,
        dpi=300,
        global_max_spectra=global_max_spectra,
        global_max_dtf=global_max_dtf,
    )
